# Repeated participant-balanced NONAN admission gate

Repeated, seed-matched validation of the 10%-mass candidate-NONAN configuration identified in notebook 20. This corrects participant window dominance. Frozen NONAN and RevalExo are not read.

In [1]:
from pathlib import Path
import sys,json,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score,balanced_accuracy_score
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import StrokeGaitInception
P=ROOT/'data'/'processed'; N=ROOT/'data'/'interim'/'nonan_gaitprint'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
x=np.concatenate([np.load(P/'validated_acceleration_magnitude_windows_float32.npy'),np.load(P/'sint_maartenskliniek_external_windows_float32.npy')]); m=pd.concat([pd.read_csv(P/'validated_window_metadata.csv'),pd.read_csv(P/'sint_maartenskliniek_external_window_metadata.csv')],ignore_index=True); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
nx=np.load(N/'candidate_healthy_enrichment_magnitude_isolated_spike_repaired.npy',mmap_mode='r'); nm=pd.read_csv(N/'candidate_healthy_enrichment_window_metadata.csv'); nm['y']=0; nm['source']='nonan_gaitprint'; nm['group']=nm.participant_key.astype(str)
cap_rng=np.random.default_rng(42); keep=np.concatenate([cap_rng.choice(v,min(64,len(v)),replace=False) for v in nm.groupby('group').indices.values()]); nx=np.asarray(nx[keep]); nm=nm.iloc[keep].reset_index(drop=True)
people=pd.concat([m[['group','source','y']].drop_duplicates(),nm[['group','source','y']].drop_duplicates()],ignore_index=True); people['stratum']=people.source+'|'+people.y.astype(str)
def weights(frame,nonan_mass):
    pcounts=frame.groupby('group').size(); cells=frame[['source','y','group']].drop_duplicates().groupby(['source','y']).size(); den=np.asarray(pd.MultiIndex.from_frame(frame[['source','y']]).map(cells),float); w=frame.group.map(1/pcounts).to_numpy()/den; w*=np.where(frame.source.eq('nonan_gaitprint'),nonan_mass,1.0); return torch.tensor(w,dtype=torch.double)
def evaluate(net,arr,meta,mean,std):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); return {'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p) if g.y.nunique()==2 else np.nan,'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5) if g.y.nunique()==2 else np.nan,'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean())}
rows=[]
for repeat in [42,137,202,404,909]:
    folds=StratifiedKFold(3,shuffle=True,random_state=repeat)
    for fold,(trp,vap) in enumerate(folds.split(people,people.stratum)):
        tg,vg=set(people.iloc[trp].group),set(people.iloc[vap].group); bt,bv=m.group.isin(tg).to_numpy(),m.group.isin(vg).to_numpy(); nt,nv=nm.group.isin(tg).to_numpy(),nm.group.isin(vg).to_numpy()
        for mode,mass in [('baseline_participant_balanced',0.),('nonan_participant_balanced_10pct',.10)]:
            torch.manual_seed(repeat*100+fold); tx,tm=x[bt],m.loc[bt].copy()
            if mass: tx,tm=np.concatenate([tx,nx[nt]]),pd.concat([tm,nm.loc[nt]],ignore_index=True)
            mean,std=tx.reshape(-1,3).mean(0),tx.reshape(-1,3).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(tm.y.to_numpy('float32')); sampler=WeightedRandomSampler(weights(tm,mass),len(tm),replacement=True,generator=torch.Generator().manual_seed(repeat*1000+fold)); dl=DataLoader(TensorDataset(z,y),128,sampler=sampler)
            net=StrokeGaitInception().to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
            for _ in range(8):
                net.train()
                for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
            net.eval()
            for name,ex,em in [('original',x[bv],m.loc[bv]),('nonan_holdout',nx[nv],nm.loc[nv])]: rows.append({'repeat':repeat,'fold':fold,'mode':mode,'evaluation':name,**evaluate(net,ex,em,mean,std)})
            del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',repeat,fold,mode)
out=pd.DataFrame(rows); out.to_csv(P/'repeated_participant_balanced_nonan_gate.csv',index=False); wide=out.pivot(index=['repeat','fold','evaluation'],columns='mode',values=['auroc','balanced_accuracy','healthy_specificity']); rng=np.random.default_rng(20260902); summary={}
for ev in ['original','nonan_holdout']:
    q=wide.xs(ev,level='evaluation')
    for metric in ['auroc','balanced_accuracy','healthy_specificity']:
        d=(q[metric]['nonan_participant_balanced_10pct']-q[metric]['baseline_participant_balanced']).dropna().to_numpy()
        if len(d):
            bs=np.array([rng.choice(d,len(d),replace=True).mean() for _ in range(10000)]); summary[f'{ev}:{metric}']={'n_paired_units':int(len(d)),'mean_delta':float(d.mean()),'bootstrap_95_ci':[float(np.quantile(bs,.025)),float(np.quantile(bs,.975))],'nonnegative_units':int((d>=0).sum())}
print(json.dumps(summary,indent=2)); (P/'repeated_participant_balanced_nonan_gate_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

device: cuda


complete 42 0 baseline_participant_balanced


complete 42 0 nonan_participant_balanced_10pct


complete 42 1 baseline_participant_balanced


complete 42 1 nonan_participant_balanced_10pct


...

complete 909 1 nonan_participant_balanced_10pct


complete 909 2 baseline_participant_balanced


complete 909 2 nonan_participant_balanced_10pct


{
  "original:auroc": {
    "n_paired_units": 15,
    "mean_delta": -0.0040603656066755455,
    "bootstrap_95_ci": [
      -0.009392459696141916,
      0.001208729341139177
    ],
    "nonnegative_units": 6
  },
  "original:balanced_accuracy": {
    "n_paired_units": 15,
    "mean_delta": 0.0054107845583153255,
    "bootstrap_95_ci": [
      -0.012041076834249228,
      0.02575781382434261
    ],
    "nonnegative_units": 8
  },
  "original:healthy_specificity": {
    "n_paired_units": 15,
    "mean_delta": -0.034254382410933745,
    "bootstrap_95_ci": [
      -0.09089847302127509,
      0.025303369977221345
    ],
    "nonnegative_units": 5
  },
  "nonan_holdout:healthy_specificity": {
    "n_paired_units": 15,
    "mean_delta": -0.0025641025641025624,
    "bootstrap_95_ci": [
      -0.01756885090218423,
      0.012250712250712252
    ],
    "nonnegative_units": 12
  }
}


883

Admission requires a non-negative lower bootstrap bound for held-out candidate healthy-specificity change, no material original-source AUROC loss, and no worsening original healthy specificity. The 15 paired resampling units quantify stability but are not independent clinical cohorts. A separate predeclared model-selection protocol is still required before touching frozen cohorts.